# Lab: Naive Bayes

## 1. Vì sao là "Naive" Bayes?

Naive Bayes là một họ thuật toán phân loại dựa trên định lý Bayes. Tên gọi *naive* (ngây thơ) đến từ một giả định lớn: ta giả sử các đặc trưng (feature) **độc lập có điều kiện** với nhau khi đã biết nhãn. Trong thực tế giả định này gần như luôn sai — nhưng kỳ lạ là Naive Bayes vẫn chạy rất tốt trên nhiều bài, đặc biệt là phân loại văn bản (spam, sentiment).

## 2. Định lý Bayes

$$
P(y \mid x) = \frac{P(x \mid y)\, P(y)}{P(x)}
$$

- $P(y)$: **prior** — xác suất tiên nghiệm của lớp $y$.
- $P(x \mid y)$: **likelihood** — xác suất quan sát thấy $x$ nếu nhãn là $y$.
- $P(x)$: **evidence** — xác suất quan sát thấy $x$ (nói chung).
- $P(y \mid x)$: **posterior** — cái ta muốn biết: xác suất nhãn là $y$ khi đã quan sát $x$.

Khi phân loại, mẫu số $P(x)$ giống nhau cho mọi lớp nên có thể bỏ:
$$
\hat{y} = \arg\max_y P(y) \cdot P(x \mid y)
$$

Đây là **Maximum A Posteriori (MAP)**.

## 3. Giả định Naive

Với $x = (x_1, x_2, \dots, x_n)$, ta giả sử:
$$
P(x \mid y) = \prod_{i=1}^{n} P(x_i \mid y)
$$

Nhờ giả định này, thay vì phải ước lượng phân phối liên kết $n$ chiều (số tham số bùng nổ), ta chỉ cần ước lượng $n$ phân phối một chiều — đơn giản hơn rất nhiều.

---

## 2b. Định lý Bayes bằng một ví dụ khiến ai cũng đoán sai

Trước khi làm toán, hãy thử câu này (một bài kiểm tra kinh điển mà **đa số bác sĩ cũng trả lời sai**):

> Một loại bệnh gặp ở **1%** dân số. Có xét nghiệm với độ nhạy 99% (người bệnh thì 99% ra dương tính) và độ đặc hiệu 95% (người khoẻ thì 95% ra âm tính). Bạn xét nghiệm và **ra dương tính**. Xác suất bạn thật sự có bệnh là bao nhiêu?

Đa số đoán "khoảng 95–99%". Đáp án đúng là **16.7%**.

![Định lý Bayes qua ví dụ xét nghiệm](images/01_dinh_ly_bayes.png)

*Cách dễ hiểu nhất là đếm người thật thay vì nhân xác suất. Trong 10.000 người: 100 người có bệnh (99 dương tính đúng), 9.900 người khoẻ (nhưng 5% trong số đó = **495 người dương tính giả**). Vậy trong 594 người nhận kết quả dương tính, chỉ 99 người thật sự có bệnh → **16.7%**.*

Điều gì tạo ra nghịch lý? **Prior.** Nhóm người khoẻ đông gấp 99 lần, nên dù tỷ lệ báo sai của họ chỉ 5%, con số tuyệt đối vẫn áp đảo số ca dương tính đúng.

$$
P(\text{bệnh} \mid +) = \frac{P(+ \mid \text{bệnh})\,P(\text{bệnh})}{P(+ \mid \text{bệnh})P(\text{bệnh}) + P(+ \mid \text{khoẻ})P(\text{khoẻ})}
= \frac{0.99 \times 0.01}{0.99 \times 0.01 + 0.05 \times 0.99} = 0.167
$$

**Bài học cho Machine Learning:** đây chính xác là lý do Naive Bayes luôn nhân likelihood với prior $P(y)$, và cũng là lý do **accuracy đánh lừa ở bài mất cân bằng** (ta sẽ gặp lại toàn bộ chuyện này ở Lab 07). Model có thể rất "chính xác" theo nghĩa likelihood mà vẫn cho posterior thấp, chỉ vì lớp đó hiếm.

### Quy tắc quyết định: MAP hay ML?

| Quy tắc | Công thức | Ý nghĩa |
|---|---|---|
| **MAP** (Maximum A Posteriori) | $\hat{y} = \arg\max_y P(y)\,P(x \mid y)$ | Có tính đến prior — **đây là cái Naive Bayes dùng** |
| **ML** (Maximum Likelihood) | $\hat{y} = \arg\max_y P(x \mid y)$ | Bỏ qua prior, tương đương giả sử mọi lớp đều xác suất bằng nhau |

Trong sklearn, prior được ước lượng từ tần suất lớp trong tập train. Muốn ép prior đều thì đặt `fit_prior=False` (Bernoulli/Multinomial NB) hoặc truyền `priors=[...]` (Gaussian NB). Việc này đáng cân nhắc khi tập train **không phản ánh đúng tỷ lệ lớp ngoài thực tế** — ví dụ bạn cố ý lấy mẫu cân bằng để train, nhưng thực tế lớp dương chỉ chiếm 1%.

## 3b. Giả định naive đánh đổi cái gì — và vì sao nó vẫn đáng

![Giả định độc lập có điều kiện](images/02_gia_dinh_naive.png)

*Trái: phân phối thật của hai feature có tương quan $\rho = 0.88$ — một dải chéo. Giữa: **những gì Naive Bayes tin** sau khi áp giả định độc lập — đám mây tròn, tương quan biến mất hoàn toàn. Phải: cái giá và cái được của sự đánh đổi này.*

### Bài toán đếm tham số

Đây là lý do thật sự khiến giả định naive tồn tại. Giả sử có $n$ feature nhị phân và $C$ lớp:

| Cách mô hình hoá | Số tham số mỗi lớp | Với $n = 30$ |
|---|---|---|
| Phân phối liên kết đầy đủ $P(x_1,\dots,x_n \mid y)$ | $2^n - 1$ | $\approx 1.07 \times 10^9$ |
| **Naive Bayes** $\prod_i P(x_i \mid y)$ | $n$ | **30** |

Hơn một tỷ tham số so với ba mươi. Không có tập dữ liệu nào trên đời đủ lớn để ước lượng một tỷ tham số một cách tin cậy — nên phân phối liên kết đầy đủ **không phải lựa chọn khả thi**, chứ không phải "lựa chọn tốt hơn nhưng đắt hơn".

### Nghịch lý: giả định sai mà kết quả vẫn đúng

Naive Bayes ước lượng xác suất **sai** (thường bị đẩy về sát 0 hoặc sát 1 một cách thái quá), nhưng **thứ tự** giữa các lớp thường vẫn đúng. Mà phân loại chỉ cần $\arg\max$ — không cần con số chính xác.

Domingos & Pazzani (1997) đã chứng minh chặt điều này: Naive Bayes tối ưu về mặt phân loại trong một lớp bài toán rộng hơn nhiều so với lớp bài toán mà giả định độc lập thật sự đúng.

> ⚠️ **Hệ quả thực tế cực kỳ quan trọng:** đừng bao giờ dùng `predict_proba()` của Naive Bayes như một xác suất đáng tin. Nó hay cho ra 0.99999 hoặc 0.00001. Nếu cần xác suất thật (để định giá rủi ro, để đặt ngưỡng theo chi phí), hãy hiệu chỉnh bằng `CalibratedClassifierCV`. Còn nếu chỉ cần nhãn thì dùng thẳng cũng được.

## 3c. Vì sao mọi thư viện làm việc trên thang LOG

Trong bài phân loại văn bản, một tài liệu có thể chứa hàng trăm từ. Nhân hàng trăm số nhỏ hơn 0.05 với nhau thì chuyện gì xảy ra?

![Underflow và thang log](images/06_lam_viec_tren_thang_log.png)

*Trái: tích các xác suất lao xuống dưới ngưỡng biểu diễn của `float64` ($\approx 2.2 \times 10^{-308}$) và **về đúng 0** chỉ sau khoảng 150 từ. Khi mọi lớp đều cho 0 thì không còn gì để so sánh — model sập. Phải: cộng logarit thì giá trị giảm tuyến tính, không bao giờ tràn.*

Vì $\log$ đơn điệu tăng, phép $\arg\max$ hoàn toàn không đổi:
$$
\hat{y} = \arg\max_y\ P(y)\prod_{i} P(x_i \mid y)
\;=\;\arg\max_y\ \Big[\log P(y) + \sum_{i} \log P(x_i \mid y)\Big]
$$

Đó là lý do sklearn cho bạn `feature_log_prob_` và `class_log_prior_` chứ không phải xác suất thô — **model học và suy luận hoàn toàn trên thang log**. Ta sẽ dùng đúng thuộc tính này ở phần "từ nào quan trọng nhất" bên dưới.

## 4. Ba biến thể phổ biến

Khác nhau ở cách mô hình hoá $P(x_i \mid y)$:

### 4.1. Bernoulli NB
Mỗi feature $x_i$ là binary (0/1). Phù hợp khi đặc trưng = "có/không có từ này trong văn bản".
$$
P(x_i \mid y) = p_{i,y}^{x_i} (1 - p_{i,y})^{1 - x_i}
$$

### 4.2. Multinomial NB
Mỗi feature $x_i$ là số đếm (count). Phù hợp khi đặc trưng = "từ này xuất hiện bao nhiêu lần" (TF, TF-IDF).
$$
P(x \mid y) \propto \prod_{i=1}^{n} p_{i,y}^{x_i}
$$

### 4.3. Gaussian NB
Mỗi feature $x_i$ liên tục, giả sử Gaussian:
$$
P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{i,y}^2}}\exp\!\left(-\frac{(x_i - \mu_{i,y})^2}{2\sigma_{i,y}^2}\right)
$$

## 5. Laplace Smoothing

Vấn đề: nếu trong tập train không có mẫu nào có $x_i = 1$ với lớp $y$, thì $P(x_i = 1 \mid y) = 0$, kéo cả tích về 0 → mô hình chết một góc.

Giải pháp: cộng thêm $\alpha$ (thường $\alpha = 1$) vào tử và mẫu khi ước lượng xác suất. Đây là Laplace smoothing (a.k.a. add-one smoothing).

## 6. Ưu nhược điểm

**Ưu:** train cực nhanh, ít tham số, hoạt động tốt với dữ liệu cao chiều và nhỏ. Là baseline mạnh cho phân loại văn bản.

**Nhược:** giả định độc lập sai — khi feature có tương quan mạnh, xác suất ước lượng có thể méo. Tuy nhiên, nếu chỉ cần `argmax` (chứ không cần xác suất chính xác), kết quả vẫn ổn.

### Ba biến thể nhìn bằng hình

![Ba biến thể Naive Bayes](images/03_ba_bien_the.png)

*Ba biến thể chỉ khác nhau ở một chỗ duy nhất: dùng phân phối gì để mô hình hoá $P(x_i \mid y)$. Bernoulli dùng phân phối nhị phân (có/không), Multinomial dùng số đếm, Gaussian dùng đường cong chuông cho giá trị liên tục.*

| | BernoulliNB | MultinomialNB | GaussianNB | CategoricalNB |
|---|---|---|---|---|
| Kiểu feature | Nhị phân 0/1 | Số đếm không âm | Số thực liên tục | Phân loại (nhiều mức) |
| $P(x_i \mid y)$ | Bernoulli | Multinomial | Chuẩn $\mathcal{N}(\mu_{iy}, \sigma^2_{iy})$ | Categorical |
| Tham số mỗi (feature, lớp) | 1 ($p$) | 1 ($p$) | 2 ($\mu, \sigma$) | $k-1$ |
| Từ **vắng mặt** có tính không? | **Có** — tính vào $(1-p)$ | Không | — | — |
| Dùng cho | Văn bản NGẮN (tweet, tiêu đề) | Văn bản DÀI, TF-IDF | Dữ liệu số đo | Dữ liệu bảng phân loại |

**Sự khác biệt tinh tế nhất giữa Bernoulli và Multinomial** nằm ở dòng "từ vắng mặt". BernoulliNB coi việc một từ **không xuất hiện** là bằng chứng có ý nghĩa (nhân thêm $1-p_{i,y}$ cho mọi từ vắng mặt trong từ điển). MultinomialNB thì bỏ qua hoàn toàn các từ vắng mặt.

Hệ quả: với văn bản dài, BernoulliNB phải nhân hàng chục nghìn thừa số $(1-p)$ cho các từ không hề liên quan → nhiễu lấn át tín hiệu. **Văn bản dài → Multinomial. Văn bản ngắn với từ điển nhỏ → Bernoulli.**

> ⚠️ **Bẫy hay gặp:** đưa feature đã `StandardScaler` (có giá trị âm) vào `MultinomialNB` sẽ báo lỗi — vì nó đòi số đếm không âm. Với dữ liệu liên tục, dùng `GaussianNB`; với TF-IDF (luôn $\ge 0$), `MultinomialNB` chạy tốt.

### Laplace smoothing nhìn kỹ hơn: $\alpha$ điều khiển cái gì

![Ảnh hưởng của alpha](images/05_laplace_smoothing.png)

*Trái: với $\alpha = 0$, hai từ chưa từng xuất hiện có xác suất **đúng bằng 0** — chỉ cần một từ như vậy trong văn bản test là cả tích về 0, và lớp đó bị loại bỏ hoàn toàn dù mọi bằng chứng khác đều ủng hộ nó. Giữa: khi tăng $\alpha$, mọi từ bị kéo dần về phân phối đều $1/V$ — model "quên" dữ liệu. Phải: công thức và cách chọn.*

$$
\hat{P}(x_i \mid y) = \frac{N_{i,y} + \alpha}{N_y + \alpha V}
$$

Cách hiểu trực quan nhất: **$\alpha$ là số lần đếm ảo** ta cộng thêm cho mỗi từ, như thể trước khi nhìn dữ liệu ta đã "thấy" mỗi từ $\alpha$ lần. Theo ngôn ngữ Bayes, đó chính là một **prior Dirichlet** đối xứng đặt lên phân phối từ, và công thức trên là posterior mean.

| $\alpha$ | Tên gọi | Hành vi |
|---|---|---|
| $0$ | Không smoothing (MLE thuần) | Xác suất 0 → sập khi gặp từ lạ |
| $(0, 1)$ | **Lidstone** | Thường tốt nhất cho text; thử $\alpha = 0.01 - 0.1$ |
| $1$ | **Laplace / add-one** | Mặc định của sklearn; an toàn nhưng hơi mạnh tay khi $V$ lớn |
| $\gg 1$ | Smoothing quá mạnh | Mọi từ gần như bằng nhau → model vô dụng |

**Vì sao $\alpha = 1$ đôi khi quá mạnh?** Nếu từ điển có $V = 50.000$ từ và một lớp chỉ có $N_y = 2.000$ token, mẫu số trở thành $2.000 + 50.000 = 52.000$ — phần "ảo" áp đảo phần "thật" gấp 25 lần. Với từ điển lớn, hãy tune $\alpha$ bằng cross-validation thay vì mặc định (đúng như Bài tập 2 ở cuối lab).

> ⚠️ `MultinomialNB(alpha=0)` sẽ báo cảnh báo và tự thay bằng một số rất nhỏ. Đừng đặt $\alpha = 0$ với dữ liệu thật.

### Gaussian NB vẽ ra ranh giới hình gì?

![Ranh giới của Gaussian NB, QDA và LDA](images/04_gaussian_nb_bien_quyet_dinh.png)

*Trái: `GaussianNB` — vì giả định các feature độc lập nên ma trận hiệp phương sai bị ép về dạng **đường chéo**, tức mọi ellipse đều song song với trục toạ độ. Giữa: `QDA` dùng hiệp phương sai đầy đủ nên ellipse **nghiêng theo dữ liệu** — bám sát hơn hẳn. Phải: `LDA` bắt hai lớp dùng chung hiệp phương sai nên ranh giới thoái hoá thành **đường thẳng**.*

Điểm gây ngạc nhiên: **ranh giới của Gaussian NB là đường cong bậc hai, không phải đường thẳng.** Lấy log tỷ số posterior của hai lớp, các số hạng $x_i^2/\sigma_{i,y}^2$ chỉ triệt tiêu khi hai lớp có cùng phương sai. Nói cách khác:

$$
\text{GaussianNB} \;=\; \text{QDA với ràng buộc hiệp phương sai đường chéo}
$$

| Model | Hiệp phương sai | Ranh giới | Số tham số |
|---|---|---|---|
| **GaussianNB** | Đường chéo, riêng từng lớp | Bậc hai (song song trục) | $2nC$ |
| **QDA** | Đầy đủ, riêng từng lớp | Bậc hai (tự do) | $C\,n(n+3)/2$ |
| **LDA** | Đầy đủ, **chung** mọi lớp | Tuyến tính | $n(n+1)/2 + nC$ |

Quy tắc chọn: ít dữ liệu và nhiều feature → GaussianNB (ít tham số nhất, khó overfit nhất). Nhiều dữ liệu và feature tương quan mạnh → QDA hoặc LDA.

# THỰC HÀNH 1: Phân loại văn bản với Bernoulli & Multinomial NB

Dữ liệu: `Education.csv` — các câu nhận xét về giáo dục, gán nhãn `positive`/`negative`. Đây là bài phân loại văn bản nhị phân điển hình.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)

np.random.seed(42)

df = pd.read_csv('Data/Education.csv')
print(f'Shape: {df.shape}')
print(f'Phân bố nhãn:\n{df["Label"].value_counts()}')
df.head()

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['Text'], df['Label'], test_size=0.2, random_state=42, stratify=df['Label'])

print(f'Train: {len(X_train_txt)}  Test: {len(X_test_txt)}')

### Hai cách biểu diễn văn bản

- **Bernoulli (binary)**: mỗi từ → 0/1 (có hay không có trong văn bản).
- **Multinomial (count)**: mỗi từ → số lần xuất hiện.

**Quan trọng**: chỉ `fit_transform` trên train, sau đó `transform` lên test — tránh data leakage (vocab của test "rò" vào train).

### Toàn cảnh đường ống phân loại văn bản

![Đường ống Naive Bayes cho văn bản](images/09_duong_ong_van_ban.png)

*Từ văn bản thô → ma trận bag-of-words → đếm và tính xác suất. Toàn bộ "huấn luyện" chỉ là **một lượt duyệt để đếm** — không lặp, không learning rate, không cần hội tụ. Đó là lý do Naive Bayes nhanh hơn mọi model khác hàng trăm lần và vẫn là baseline chuẩn cho bài text.*

Vài lựa chọn tiền xử lý ảnh hưởng mạnh đến kết quả:

| Lựa chọn | Tác dụng | Lưu ý |
|---|---|---|
| `lowercase=True` (mặc định) | "Good" và "good" gộp làm một | Gần như luôn nên bật |
| `stop_words='english'` | Bỏ "the", "is", "at"… | Với **tiếng Việt** phải tự cung cấp danh sách |
| `ngram_range=(1,2)` | Thêm cụm 2 từ → bắt được "không tốt" | Từ điển phình to, cần nhiều dữ liệu hơn |
| `min_df=2` | Bỏ từ chỉ xuất hiện 1 lần | Giảm nhiễu và giảm chiều rất hiệu quả |
| `TfidfVectorizer` | Hạ trọng số từ phổ biến | Thường tốt hơn raw count (xem Bài tập 3) |

> ⚠️ **Tiếng Việt cần tách từ trước.** `CountVectorizer` tách theo khoảng trắng nên "học máy" thành hai token rời rạc, mất nghĩa. Với dữ liệu tiếng Việt thật, hãy dùng `underthesea` hoặc `pyvi` để word-segment trước, rồi mới vectorize.

In [ ]:
# Bernoulli: binary features
vec_bin = CountVectorizer(binary=True, stop_words='english')
X_train_bin = vec_bin.fit_transform(X_train_txt)
X_test_bin  = vec_bin.transform(X_test_txt)

# Multinomial: count features
vec_cnt = CountVectorizer(stop_words='english')
X_train_cnt = vec_cnt.fit_transform(X_train_txt)
X_test_cnt  = vec_cnt.transform(X_test_txt)

print(f'Số feature (Bernoulli): {X_train_bin.shape[1]}')
print(f'Số feature (Multinomial): {X_train_cnt.shape[1]}')
print('5 từ ngẫu nhiên trong vocab:', np.random.choice(vec_cnt.get_feature_names_out(), 5).tolist())

In [ ]:
# Train cả hai
bnb = BernoulliNB()
bnb.fit(X_train_bin, y_train)
y_pred_bnb = bnb.predict(X_test_bin)

mnb = MultinomialNB()
mnb.fit(X_train_cnt, y_train)
y_pred_mnb = mnb.predict(X_test_cnt)

print(f'Bernoulli NB    accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}%')
print(f'Multinomial NB  accuracy: {accuracy_score(y_test, y_pred_mnb)*100:.2f}%')
print()
print('Báo cáo chi tiết — Multinomial NB:')
print(classification_report(y_test, y_pred_mnb))

### ROC curve

ROC vẽ True Positive Rate (recall) theo False Positive Rate khi quét ngưỡng. AUC = diện tích dưới đường cong, càng gần 1 càng tốt; 0.5 = đoán mò.

Lưu ý: **AUC < 0.5 nghĩa là model tệ hơn cả đoán mò**, nhưng có thể "đảo ngược" predict (lật label) để được > 0.5 — chứ không phải không cứu được.

In [ ]:
# Lấy xác suất class "positive" để vẽ ROC
pos_idx = list(mnb.classes_).index('positive')
proba_mnb = mnb.predict_proba(X_test_cnt)[:, pos_idx]

fpr, tpr, _ = roc_curve(y_test, proba_mnb, pos_label='positive')
auc_score = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Multinomial NB (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Đoán mò (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Từ nào quan trọng nhất với mỗi lớp?

Multinomial NB lưu `feature_log_prob_[class][token]` = log P(token | class). Tỷ số log giữa hai lớp cho biết từ nào "thiên vị" về phía nào.

In [ ]:
vocab = vec_cnt.get_feature_names_out()
log_p_pos = mnb.feature_log_prob_[pos_idx]
log_p_neg = mnb.feature_log_prob_[1 - pos_idx]
log_ratio = log_p_pos - log_p_neg     # >0 → ngả về positive, <0 → ngả về negative

top_pos = np.argsort(log_ratio)[-10:][::-1]
top_neg = np.argsort(log_ratio)[:10]

print('Top 10 từ thiên vị POSITIVE:')
for i in top_pos:
    print(f'  {vocab[i]:20s}  log-ratio = {log_ratio[i]:+.3f}')
print('\nTop 10 từ thiên vị NEGATIVE:')
for i in top_neg:
    print(f'  {vocab[i]:20s}  log-ratio = {log_ratio[i]:+.3f}')

In [ ]:
# Naive Bayes rất tự tin — nhưng có đáng tin không?
# So sánh mức độ hiệu chỉnh xác suất giữa Multinomial NB và Logistic Regression.
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

lr_txt = LogisticRegression(max_iter=2000).fit(X_train_cnt, y_train)
proba_lr = lr_txt.predict_proba(X_test_cnt)[:, pos_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Phân phối xác suất model xuất ra
axes[0].hist(proba_mnb, bins=20, alpha=0.65, label='Multinomial NB', color='#dc2626')
axes[0].hist(proba_lr,  bins=20, alpha=0.65, label='Logistic Regression', color='#2563eb')
axes[0].set_xlabel('Xác suất dự đoán cho lớp positive')
axes[0].set_ylabel('Số mẫu'); axes[0].legend()
axes[0].set_title('NB dồn hết về 0 và 1 — "tự tin thái quá"')

# Đường hiệu chỉnh (reliability diagram)
for proba, name, col in [(proba_mnb, 'Multinomial NB', '#dc2626'),
                          (proba_lr, 'Logistic Regression', '#2563eb')]:
    n_bins = min(5, len(np.unique(np.round(proba, 2))))
    frac, mean_pred = calibration_curve(
        (y_test == 'positive').astype(int), proba, n_bins=n_bins, strategy='quantile')
    axes[1].plot(mean_pred, frac, 'o-', color=col, label=name)
axes[1].plot([0, 1], [0, 1], 'k--', label='hiệu chỉnh hoàn hảo')
axes[1].set_xlabel('Xác suất trung bình model nói')
axes[1].set_ylabel('Tỷ lệ thực sự là positive')
axes[1].legend(fontsize=8); axes[1].set_title('Reliability diagram')

plt.tight_layout(); plt.show()

print('Quan sát: Naive Bayes gần như chỉ xuất ra 0.00 hoặc 1.00.')
print('Nguyên nhân: giả định độc lập khiến nó NHÂN quá nhiều bằng chứng tương quan')
print('  -> log-odds bị thổi phồng -> sigmoid/softmax bão hoà.')
print('Kết luận: dùng NB để LẤY NHÃN thì ổn; cần XÁC SUẤT thì phải CalibratedClassifierCV.')
print()
print('Lưu ý: dataset Education chỉ có ~10 mẫu test nên biểu đồ rất nhiễu.')
print('Hãy chạy lại thí nghiệm này trên Breast Cancer hoặc 20-newsgroups để thấy rõ hơn.')

# THỰC HÀNH 2: Gaussian NB trên dữ liệu thuốc (drug200)

Dữ liệu `drug200.csv` có cả feature liên tục (Age, Na_to_K) và rời rạc (Sex, BP, Cholesterol). Mục tiêu: dự đoán loại thuốc phù hợp.

**Lưu ý quan trọng**: Gaussian NB *giả định feature là liên tục, phân phối chuẩn*. Nếu ép feature one-hot (binary) vào Gaussian NB, ta đang vi phạm giả định — kết quả vẫn chạy nhưng không phải là cách dùng đúng. Trong bài này ta dùng Gaussian NB trên feature đã encode chỉ để minh hoạ, kèm thảo luận giới hạn.

In [ ]:
from sklearn.preprocessing import LabelEncoder

drug = pd.read_csv('Data/drug200.csv')
print(drug.head())
print(f'\nShape: {drug.shape}, classes: {drug["Drug"].unique()}')

In [ ]:
# Encode các cột rời rạc
df_enc = drug.copy()
for col in ['Sex', 'BP', 'Cholesterol']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

X = df_enc.drop('Drug', axis=1).values
y = LabelEncoder().fit_transform(df_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
print(f'Gaussian NB accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
drug_names = LabelEncoder().fit(df_enc['Drug']).classes_
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(drug_names); ax.set_yticklabels(drug_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix — Gaussian NB trên drug200')
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

---

## 7. Naive Bayes trong bức tranh lớn: mô hình SINH MẪU

![Sinh mẫu so với phân biệt](images/07_sinh_mau_vs_phan_biet.png)

*Trái: đường học (learning curve) của Naive Bayes và Logistic Regression trên cùng bộ dữ liệu. Naive Bayes **thắng rõ khi ít dữ liệu**, rồi bị vượt qua khi dữ liệu nhiều lên. Phải: bảng so sánh hai trường phái.*

Đây là kết quả kinh điển của **Ng & Jordan (2001)**, và nó giải thích chính xác khi nào nên chọn Naive Bayes:

- **Naive Bayes** học $P(x, y)$ — mô hình hoá *toàn bộ* cách dữ liệu được sinh ra. Giả định độc lập tạo ra **bias cao nhưng variance thấp**: nó hội tụ rất nhanh (chỉ cần $O(\log n)$ mẫu) nhưng hội tụ tới một nghiệm **hơi sai** vì giả định sai.
- **Logistic Regression** học thẳng $P(y \mid x)$ — chỉ quan tâm ranh giới. **Bias thấp, variance cao**: cần nhiều dữ liệu hơn ($O(n)$ mẫu) nhưng cuối cùng đạt nghiệm tốt hơn.

> **Quy tắc bỏ túi:** ít dữ liệu, nhiều feature (đặc biệt là text) → thử Naive Bayes trước. Nhiều dữ liệu → Logistic Regression / SVM / ensemble sẽ vượt lên. Và **luôn chạy Naive Bayes làm baseline** — nó tốn vài giây, nếu model phức tạp của bạn không thắng nổi nó thì có gì đó sai.

## 8. Điểm yếu chí mạng: feature trùng lặp bị đếm nhiều lần

![Tương quan phá Naive Bayes](images/08_tuong_quan_pha_naive_bayes.png)

*Thí nghiệm: nhân bản **cùng một feature** nhiều lần. Thông tin không hề tăng thêm chút nào, nhưng accuracy của Naive Bayes tụt dần, trong khi Logistic Regression gần như đứng yên.*

Nguyên nhân rõ ràng khi viết ra: nếu $x_1$ được sao thành ba bản giống hệt, Naive Bayes tính
$$
P(x_1 \mid y)\cdot P(x_1' \mid y)\cdot P(x_1'' \mid y) = P(x_1 \mid y)^3
$$
**Một bằng chứng bị tính ba lần.** Model trở nên tự tin thái quá và nghiêng hẳn về phía mà $x_1$ ủng hộ. Logistic Regression thì tự động chia trọng số cho các bản sao nên tổng đóng góp giữ nguyên.

**Cách phòng tránh trong thực tế:**
1. Vẽ ma trận tương quan trước khi dùng NB; bỏ bớt feature có $|\rho| > 0.9$.
2. Với text: dùng TF-IDF thay raw count (giảm bớt hiệu ứng đếm trùng của từ phổ biến).
3. Cẩn thận khi one-hot: cột one-hot của cùng một biến **loại trừ lẫn nhau**, tức tương quan âm hoàn hảo — đây là vi phạm giả định độc lập rõ rệt. Dùng `CategoricalNB` cho dữ liệu phân loại thay vì one-hot + GaussianNB.
4. Nếu buộc phải giữ feature tương quan, hãy giảm chiều bằng PCA trước (các thành phần chính trực giao nhau).

### Bảng tổng kết: khi nào dùng Naive Bayes

| Nên dùng khi | Nên tránh khi |
|---|---|
| Phân loại văn bản, lọc spam, phân tích cảm xúc | Feature tương quan mạnh với nhau |
| Dữ liệu rất nhiều chiều, rất thưa | Cần **xác suất** đáng tin (chưa hiệu chỉnh) |
| Tập train nhỏ | Có nhiều dữ liệu và cần độ chính xác tối đa |
| Cần baseline trong vài giây | Quan hệ giữa các feature chính là tín hiệu (ví dụ ảnh) |
| Cần train lại liên tục (online, streaming) | Feature liên tục lệch xa phân phối chuẩn |
| Dữ liệu có ô trống rải rác | |

## Tổng kết

1. **Định lý Bayes** + giả định độc lập có điều kiện = Naive Bayes.
2. Ba biến thể chính: Bernoulli (binary), Multinomial (count), Gaussian (liên tục).
3. **Laplace smoothing** ($\alpha$) giúp tránh xác suất = 0.
4. Trong bài text, Multinomial NB thường tốt hơn Bernoulli NB nếu document đủ dài.
5. Naive Bayes là *baseline cực mạnh* — train rất nhanh, kết quả ổn, nên luôn thử trước khi dùng model phức tạp.

## Tránh các bẫy
- KHÔNG `fit_transform` trên test — chỉ `transform`.
- KHÔNG fit Gaussian NB trên feature one-hot mà không cảnh báo về giả định bị vi phạm.
- KHI dataset nhỏ, accuracy trên test rất nhiễu — nên dùng cross-validation.

# BÀI TẬP VỀ NHÀ

## Bài 1: Cross-validation cho text classifier
Dataset Education chỉ có 52 dòng — accuracy trên 1 lần chia rất nhiễu. Hãy:
1. Dùng `StratifiedKFold(n_splits=5)` để cross-validate cả Bernoulli NB và Multinomial NB.
2. Báo cáo accuracy trung bình ± std cho mỗi model.
3. Model nào ổn định hơn?

*Gợi ý:* `from sklearn.model_selection import cross_val_score; cross_val_score(model, X, y, cv=5)`.

## Bài 2: Ảnh hưởng của Laplace smoothing
Train Multinomial NB trên Education với `alpha ∈ {0.01, 0.1, 1, 10, 100}`. Vẽ accuracy theo alpha. Quan sát: alpha quá nhỏ và quá lớn đều tệ — vì sao?

*Gợi ý:* `MultinomialNB(alpha=...)`. Alpha lớn = smoothing mạnh = mô hình "quên" data.

## Bài 3: TF-IDF thay vì raw count
Thay `CountVectorizer` bằng `TfidfVectorizer`. Train Multinomial NB. So sánh accuracy với raw count. TF-IDF có giúp không?

## Bài 4: CategoricalNB cho drug200
Sklearn có `CategoricalNB` — biến thể *đúng* cho feature rời rạc. Dùng nó trên drug200 (sau khi encode). So sánh với Gaussian NB. Cái nào tốt hơn? Vì sao?

*Gợi ý:* `from sklearn.naive_bayes import CategoricalNB`. Lưu ý: feature liên tục Age, Na_to_K cần discretize trước (`pd.cut` hoặc `KBinsDiscretizer`).

## Bài 5: Phân tích sai
Trên drug200, in ra 5 mẫu Gaussian NB đoán sai. Đặc trưng của chúng có gì đặc biệt? (Age cực, Na_to_K cao bất thường?)

*Gợi ý:* `mask = y_pred != y_test; X_test[mask][:5]`.